# YOLOv8s — Test-set evaluation

Оценка чекпоинта `weak` (из augmentation_study) на `test`-сплите BrackishMOT.
Параметры валидации совпадают с `train_yolo26s.ipynb` — честное сравнение архитектур.

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Working dir: {Path.cwd()}")

import torch
print(f"PyTorch:     {torch.__version__}")
print(f"CUDA:        {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## Поиск чекпоинта

ultralytics автоинкрементит имя рана (`weak`, `weak2`, ...) — берём самый свежий weak-ран по mtime.

In [ ]:
candidates = sorted(
    Path("runs/detect/runs/augmentation_study").glob("weak*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError(
        "Не найден вес v8s в runs/detect/runs/augmentation_study/weak*/weights/best.pt. "
        "Укажи путь вручную."
    )
WEIGHTS = candidates[-1]
print(f"Using: {WEIGHTS}")

## Evaluate on test set

In [ ]:
best = YOLO(str(WEIGHTS))

metrics = best.val(
    data="configs/dataset.yaml",
    split="test",
    batch=1,
    device=0,
    plots=True,
)

print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

print(f"\nInference speed: {metrics.speed}")

print("\nPer-class AP@50:")
for i, name in metrics.names.items():
    print(f"  {name:12s}  {metrics.box.ap50[i]:.4f}")